### ESEMPIO TIME SERIES Gini 

In [ ]:
import pandas as pd
from pathlib import Path 
import geopandas as geopd
import plotly.express as px  
import numpy as np 
import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / "utils.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
    
from utils import get_s3, get_mapping

# CARICAMENTO DATI 

In [ ]:
vodafone_attendences_new = pd.read_csv(get_s3("vodafone_attendences_new.csv"))

geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")
json_apt = get_mapping("map_comuni_into_apt.json")

In [ ]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_attendences_new['date'] = pd.to_datetime(vodafone_attendences_new['date'])
vodafone_attendences_new = vodafone_attendences_new.sort_values('date').reset_index(drop=True)

vodafone_attendences_merged_comuni = vodafone_attendences_new[
    vodafone_attendences_new["locType"] == "TN_MKT_AL_3"
]
vodafone_attendences_merged_comuni["comune"] = vodafone_attendences_merged_comuni["locId"].astype(str).map(location_map)
vodafone_attendences_merged_comuni["ID_COMUNE"] = vodafone_attendences_merged_comuni["comune"].map(json_vodafone)
# Fix vigo and pozza 
mask = vodafone_attendences_merged_comuni["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_attendences_merged_comuni.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA" 
vodafone_attendences_merged_comuni.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_attendences_merged_comuni.index[mask]  
)

tourists_attendences = vodafone_attendences_merged_comuni[
    vodafone_attendences_merged_comuni['userProfile'] == 'TOURIST'
].groupby(['date', 'comune']).agg({'value': 'sum', 'ID_COMUNE':'first'}).reset_index()

t = tourists_attendences.copy()

In [ ]:
pivot_raw = t.pivot(index="date", columns="comune", values="value")
min_date, max_date = pivot_raw.index.min(), pivot_raw.index.max()
pivot_raw = pivot_raw.reindex(pd.date_range(min_date, max_date, freq="D"))
pivot_tourists = pivot_raw.fillna(0) # 32 comuni tt.isna().sum()[tt.isna().sum() > 0].sort_values()

print(min_date, max_date)

In [ ]:
diag = pd.DataFrame({
    "giorni_NaN": pivot_raw.isna().sum(),
    "giorni_zero_nei_dati": (pivot_raw == 0).sum(),
    "media_giorno": pivot_raw.mean(),
})
diag[diag['giorni_NaN'] > 0].sort_values('giorni_NaN', ascending=False)

In [ ]:
def gini_index(values):
    x = np.asarray(values, dtype=float)
    if x.size < 2 or np.isnan(x).any() or (x < 0).any() or x.sum() == 0:
        return np.nan
    x = np.sort(x); n = x.size
    g = (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())
    return g * n / (n - 1)   # normalizzato 

def gini_window(pivot_tourists, start, n_months, rolling=None):
    """Gini per comune su [start, start + n_months), con dati giornalieri.
    rolling: finestra opzionale di smoothing in giorni (per esempio 7 o 14)."""

    end = pd.Timestamp(start) + pd.DateOffset(months=n_months) - pd.Timedelta(days=1)
    assert pivot_tourists.index.min() <= pd.Timestamp(start) and end <= pivot_tourists.index.max(), "finestra fuori dai dati"

    x = pivot_tourists.rolling(rolling, center=True).mean() if rolling else pivot_tourists
    return x.loc[pd.Timestamp(start):end].apply(gini_index).rename(f"{pd.Timestamp(start):%Y-%m-%d}+{n_months}m")

def plot_gini_map(gini_series, title, range_color=None, width=1000, height=800):
    gdf = geojson_comuni_json_data.copy()
    gdf["comune_upper"] = gdf["name"].astype(str).str.strip().str.upper()
    gdf.loc[
        gdf["comune_upper"].isin(["VIGO DI FASSA", "POZZA DI FASSA"]),
        "comune_upper",
    ] = "SAN GIOVANNI DI FASSA"

    gini_df = (
        gini_series.rename("gini")
        .rename_axis("comune")
        .reset_index()
    )
    gini_df["_comune_key"] = gini_df["comune"].astype(str).str.strip().str.upper()

    missing = set(gini_df["_comune_key"]) - set(gdf["comune_upper"])
    if missing:
        print("Comuni nel Gini senza poligono:", missing)

    map_gini = gdf.merge(
        gini_df[["_comune_key", "gini"]],
        left_on="comune_upper",
        right_on="_comune_key",
        how="left",
        validate="many_to_one",
    )

    # ID esplicito, presente sia nel GeoJSON sia in locations.
    map_gini["_map_id"] = map_gini.index.astype(str)

    fig = px.choropleth(
        map_gini,
        geojson=map_gini.__geo_interface__,
        locations="_map_id",
        featureidkey="properties._map_id",
        color="gini",
        hover_name="name",
        hover_data={"gini": ":.3f", "comune_upper": False, "_map_id": False},
        color_continuous_scale="YlOrRd",
        range_color=range_color,
        title=title,
        template="plotly_white",
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(
        width=width,
        height=height,
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        coloraxis_colorbar_title="Gini",
    )
    return fig


s = pivot_tourists.isna().sum()
s[s>0]

In [ ]:
# pivot_tourists = pivot_tourists.fillna(0)   # i giorni mancanti sono su comuni molto piccoli

In [ ]:
mesi = 12 
start = "2023-01-01"
g = gini_window(pivot_tourists, start,mesi)
# g_estate = pd.DataFrame({y: gini_window(pivot_tourists, f"{y}-06-15", 3) for y in [2023, 2024, 2025]})
# g_estate["media"] = g_estate.mean(axis=1)
g

In [ ]:
plot_gini_map(g, "Gini 2024").show()
g

In [ ]:
comuni_top = g.sort_values(ascending = False).head(15)
comuni_nop = g.sort_values(ascending = False).tail(15)
comuni_nop

In [ ]:
comune_test = "CALCERANICA AL LAGO"
ts_comune = pivot_tourists[comune_test]

fig_smooth = go.Figure()

start = pd.to_datetime(start)
end = start + pd.DateOffset(months=mesi)

ts_comune_filtered = ts_comune[(ts_comune.index >= start) & (ts_comune.index < end )]
fig_smooth.add_trace(go.Scatter(
    x=ts_comune_filtered.index, y=ts_comune_filtered,
    mode='lines', name='Dati Grezzi (Giornalieri)',
))

fig_smooth.update_layout(
    title=f"Smoothing - Comune di {comune_test}",
    xaxis_title="Data",
    yaxis_title="Presenze Turistiche",
    template="plotly_white",
    hovermode="x unified"
)
fig_smooth.show()

In [ ]:
g_2024 = gini_window(pivot_tourists, "2024-06-15", 3, rolling=7)
plot_gini_map(g_2024, "Gini estate 2024 (15 giu - 14 set, rolling 7 gg)").show()

# media 2023-2025 (escludendo il 2022 finché non verifichi la copertura)
g_estate = pd.DataFrame({y: gini_window(pivot_tourists, f"{y}-06-15", 3) for y in [2023, 2024, 2025]})
plot_gini_map(g_estate.mean(axis=1), "Gini estate, media 2023-2025").show()

# per confrontare più mappe, stessa scala di colore
# ...existing code...

# Ripristina l'indice se era già stato spostato in una colonna
if "comune" in g_estate.columns:
    g_estate = g_estate.set_index("comune")

# Usa la stessa scala per confrontare le mappe
rng = (0, float(g_estate.max().max()))

for y in g_estate.columns:
    plot_gini_map(
        g_estate[y],
        f"Gini estate {y}",
        range_color=rng,
    ).show()

## trash 

In [ ]:
def gini_period(s, start, end, freq="D", min_coverage=0.8, min_buckets=6):
    """
    Gini per comune nel periodo [start, end] (estremi inclusi).

    s            : DataFrame giorno x comune, valori grezzi, NaN per i giorni mancanti
    freq         : granularità di aggregazione: "D" giornaliera, "W" settimanale, "MS" mensile
                   (media giornaliera per ogni bucket)
    rolling      : finestra (giorni) di smoothing opzionale, applicata PRIMA di tagliare
                   il periodo, così non ci sono effetti di bordo
    min_coverage : quota minima di giorni con dato nel periodo, sotto cui il risultato è NaN
    min_buckets  : numero minimo di valori su cui calcolare il Gini (avviso se inferiore)
    """
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    raw = s.loc[start:end]

    x = raw
    if freq != "D":
        x = x.resample(freq).mean()

    if len(x) < min_buckets:
        print(f"Attenzione: solo {len(x)} valori per il Gini ({freq}) nel periodo "
              f"{start.date()} - {end.date()}: usa una granularità più fine.")

    coverage = raw.notna().mean()
    out = x.apply(lambda c: gini_index(c.dropna().values))
    return out.where(coverage >= min_coverage).rename(f"gini_{start.date()}_{end.date()}")


# un periodo
g = gini_period(pivot_tourists, "2024-01-01", "2024-12-31", freq="MS")

# stesso periodo anno per anno (e media)
g_years = pd.concat(
    {y: gini_period(pivot_tourists, f"{y}-01-01", f"{y}-12-31", freq="MS") for y in [2023, 2024, 2025]},
    axis=1,
)
g_mean = g_years.mean(axis=1)

# stagione specifica, qui l'inverno 2023/24, con dati giornalieri smussati
g_inv = gini_period(pivot_tourists, "2023-12-01", "2024-03-31", freq="D")

# più periodi con nomi (per esempio le finestre di picco trovate nell'altro notebook)
periodi = {
    "inverno_23_24": ("2023-12-01", "2024-03-31"),
    "estate_24":     ("2024-06-15", "2024-09-15"),
}
g_periodi = pd.concat(
    {k: gini_period(pivot_tourists, a, b, freq="D") for k, (a, b) in periodi.items()},
    axis=1,
)

g_periodi.sort_values(by= 'estate_24')

In [ ]:
tot = pivot_tourists.sum(axis=1)
print(tot[tot == 0])                         # giorni vuoti per TUTTI i comuni = buchi nei dati
print(tot.groupby(tot.index.year).apply(lambda x: (x == 0).sum()))


In [ ]:
win = pivot_tourists.loc["2024-06-15":"2024-09-14"]
chk = pd.DataFrame({"gini": gini_window(pivot_tourists, "2024-06-15", 3),
                    "media_giorno": win.mean(), "giorni_zero": (win == 0).sum()})
chk.sort_values("media_giorno").head(10)

In [ ]:
cfg = pd.DataFrame({
    "raw": gini_window(pivot_tourists, "2024-06-15", 3),
    "r7":  gini_window(pivot_tourists, "2024-06-15", 3, rolling=7),
    "r14": gini_window(pivot_tourists, "2024-06-15", 3, rolling=14),
})
cfg.corr(method="spearman")

In [ ]:
g_estate[2024].sort_values().head(20)